<a href="https://colab.research.google.com/github/NaziaAfreen015/CSC791-DLBA/blob/main/CSC791_DLBA_VGG_FGSM_Transfer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transfer-based black-box adversarial attack between the dense and pruned models

In [1]:
import tensorflow as tf
import torchvision.models as models
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
import requests
from io import BytesIO
import torch
import torch.optim as optim
import torch.nn.utils.prune as prune
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
import torchvision

import copy
import math
from typing import List, Tuple
import time


import cv2
import matplotlib.pyplot as plt

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
def get_train_transform(transform_size=224):
  train_transform = transforms.Compose([
      transforms.Resize((transform_size, transform_size)),
      transforms.RandomHorizontalFlip(),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
  ])
  return train_transform

def get_test_transform(transform_size):
  test_transform = transforms.Compose([
      transforms.Resize((transform_size, transform_size)),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
  ])
  return test_transform


In [5]:
def get_train_val_split(dataset='10', val_ratio=0.1, seed=42):
  if dataset == '10':
    full_train_for_indices = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=None)
  else:
    full_train_for_indices = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=None)

  num_train = len(full_train_for_indices)
  num_val = int(val_ratio * num_train)
  num_train_split = num_train - num_val

  generator = torch.Generator().manual_seed(seed)
  permuted_indices = torch.randperm(num_train, generator=generator).tolist()

  train_indices = permuted_indices[:num_train_split]
  val_indices = permuted_indices[num_train_split:]

  return train_indices, val_indices


In [6]:
# from torchvision.datasets.vision import data
def load_dataset(dataset='10', val_ratio=0.1, seed=42, batch_size=64):
  train_transform = get_train_transform(transform_size=224)
  test_transform = get_test_transform(transform_size=224)

  if dataset == '10':
    train_indices, val_indices = get_train_val_split(dataset=dataset, val_ratio=val_ratio, seed=seed)

    trainset_full = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=train_transform)
    valset_full = datasets.CIFAR10(root="./data", train=True, download=False, transform=test_transform)

    trainset = Subset(trainset_full, train_indices)
    valset = Subset(valset_full, val_indices)

    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    # Download and load test dataset
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

    print('CIFAR-10 dataset downloaded and loaded successfully.')

  elif dataset == '100':
    train_indices, val_indices = get_train_val_split(dataset=dataset, val_ratio=val_ratio, seed=seed)

    trainset_full = torchvision.datasets.CIFAR100(root='./data', train=True, download=False, transform=train_transform)
    valset_full = datasets.CIFAR100(root="./data", train=True, download=False, transform=test_transform)

    trainset = Subset(trainset_full, train_indices)
    valset = Subset(valset_full, val_indices)

    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    # Download and load test dataset
    testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=test_transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    classes = ('beaver', 'dolphin', 'otter', 'seal', 'whale', 'aquarium fish', 'flatfish', 'ray', 'shark', 'trout', 'orchids', 'poppies', 'roses', 'sunflowers', 'tulips', 'bottles', 'bowls', 'cans', 'cups', 'plates', 'apples', 'mushrooms', 'oranges', 'pears', 'sweet peppers', 'clock', 'computer keyboard', 'lamp', 'telephone', 'television', 'bed', 'chair', 'couch', 'table', 'wardrobe', 'bee', 'beetle', 'butterfly', 'caterpillar', 'cockroach', 'bear', 'leopard', 'lion', 'tiger', 'wolf', 'bridge', 'castle', 'house', 'road', 'skyscraper', 'cloud', 'forest', 'mountain', 'plain', 'sea', 'camel', 'cattle', 'chimpanzee', 'elephant', 'kangaroo', 'fox', 'porcupine', 'possum', 'raccoon', 'skunk', 'crab', 'lobster', 'snail', 'spider', 'worm', 'baby', 'boy', 'girl', 'man', 'woman', 'crocodile', 'dinosaur', 'lizard', 'snake', 'turtle', 'hamster', 'mouse', 'rabbit', 'shrew', 'squirrel', 'maple', 'oak', 'palm', 'pine', 'willow', 'bicycle', 'bus', 'motorcycle', 'pickup truck', 'train', 'lawn-mower', 'rocket', 'streetcar', 'tank', 'tractor')
    print('CIFAR-100 dataset downloaded and loaded successfully.')

  return trainset, trainloader, testset, testloader, valset, valloader, classes


In [7]:
def customize_model(model, num_classes):
  model.classifier[6] = nn.Linear(4096, num_classes)
  return model


In [8]:
def get_model(num_classes, path=''):
  if path != '':
    model = models.vgg16(weights=None)
    model = customize_model(model, num_classes)
    print(f'Loading model from path {path}')
    model.load_state_dict(torch.load(path, map_location=device))
  else:
    model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
    model = customize_model(model, num_classes)
  model = model.to(device)
  return model

In [9]:
def get_normalized_bounds(mean, std, device):
    """
    What this does:
    - Converts the valid raw pixel range [0, 1] into normalized-space bounds
    - These bounds are used to clamp adversarial images correctly
    """
    mean = mean.to(device).view(1, 3, 1, 1)
    std = std.to(device).view(1, 3, 1, 1)

    clamp_min = (0.0 - mean) / std
    clamp_max = (1.0 - mean) / std
    return clamp_min, clamp_max


def fgsm_attack_normalized(images, epsilon, data_grad, std):
    """
    What this does:
    - Applies FGSM to normalized images
    - Interprets epsilon in raw pixel space, then rescales it channel-wise
    - Returns adversarial images in normalized space
    """
    std = std.to(images.device).view(1, 3, 1, 1)
    epsilon_normalized = epsilon / std
    adv_images = images + epsilon_normalized * data_grad.sign()
    return adv_images


def generate_transfer_fgsm_batch(source_model, images, labels, epsilon, device, mean, std):
    """
    What this does:
    - Uses the source model to compute input gradients
    - Creates FGSM adversarial examples from those gradients
    - These adversarial examples can then be tested on another model
    """
    criterion = nn.CrossEntropyLoss()

    mean = mean.to(device)
    std = std.to(device)
    clamp_min, clamp_max = get_normalized_bounds(mean, std, device)

    images = images.to(device).detach().clone()
    labels = labels.to(device)

    images.requires_grad = True

    outputs = source_model(images)
    loss = criterion(outputs, labels)

    source_model.zero_grad()
    loss.backward()

    data_grad = images.grad.detach()
    adv_images = fgsm_attack_normalized(images, epsilon, data_grad, std)

    adv_images = torch.max(torch.min(adv_images, clamp_max), clamp_min)
    adv_images = adv_images.detach()

    return adv_images


def evaluate_transfer_attack(source_model, target_model, testloader, epsilon, device, mean, std, save_per_batch=0):
    """
    What this does:
    - Creates FGSM adversarial examples on the source model
    - Evaluates those same adversarial examples on the target model
    - Reports:
        1. clean accuracy of source model
        2. clean accuracy of target model
        3. adversarial accuracy on source model
        4. adversarial accuracy on target model
        5. transfer fooling rate on initially correct target examples
    - Optionally saves a few successful transferred examples per batch
    """
    source_model.eval()
    target_model.eval()

    total = 0
    source_clean_correct = 0
    target_clean_correct = 0
    source_adv_correct = 0
    target_adv_correct = 0

    target_initially_correct = 0
    target_transfer_success = 0

    saved_examples = []

    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)

        # Clean predictions
        with torch.no_grad():
            source_clean_outputs = source_model(images)
            target_clean_outputs = target_model(images)

            source_clean_preds = source_clean_outputs.argmax(dim=1)
            target_clean_preds = target_clean_outputs.argmax(dim=1)

        # Generate adversarial examples using SOURCE model gradients
        adv_images = generate_transfer_fgsm_batch(
            source_model=source_model,
            images=images,
            labels=labels,
            epsilon=epsilon,
            device=device,
            mean=mean,
            std=std
        )

        # Evaluate both models on those same adversarial images
        with torch.no_grad():
            source_adv_outputs = source_model(adv_images)
            target_adv_outputs = target_model(adv_images)

            source_adv_preds = source_adv_outputs.argmax(dim=1)
            target_adv_preds = target_adv_outputs.argmax(dim=1)

        batch_size = labels.size(0)
        total += batch_size

        source_clean_correct += (source_clean_preds == labels).sum().item()
        target_clean_correct += (target_clean_preds == labels).sum().item()

        source_adv_correct += (source_adv_preds == labels).sum().item()
        target_adv_correct += (target_adv_preds == labels).sum().item()

        # Transfer fooling rate: among target-clean-correct examples, how many are flipped by transferred attack?
        target_correct_mask = (target_clean_preds == labels)
        target_transfer_success_mask = target_correct_mask & (target_adv_preds != labels)

        target_initially_correct += target_correct_mask.sum().item()
        target_transfer_success += target_transfer_success_mask.sum().item()

        if save_per_batch > 0:
            idxs = target_transfer_success_mask.nonzero(as_tuple=True)[0][:save_per_batch]
            for idx in idxs:
                saved_examples.append({
                    "clean_image": images[idx].detach().cpu(),
                    "adv_image": adv_images[idx].detach().cpu(),
                    "label": labels[idx].item(),
                    "source_clean_pred": source_clean_preds[idx].item(),
                    "source_adv_pred": source_adv_preds[idx].item(),
                    "target_clean_pred": target_clean_preds[idx].item(),
                    "target_adv_pred": target_adv_preds[idx].item(),
                })

    source_clean_acc = source_clean_correct / total
    target_clean_acc = target_clean_correct / total
    source_adv_acc = source_adv_correct / total
    target_adv_acc = target_adv_correct / total

    target_transfer_fooling_rate = (
        target_transfer_success / target_initially_correct
        if target_initially_correct > 0 else 0.0
    )

    return {
        "source_clean_acc": source_clean_acc,
        "target_clean_acc": target_clean_acc,
        "source_adv_acc": source_adv_acc,
        "target_adv_acc": target_adv_acc,
        "target_transfer_fooling_rate": target_transfer_fooling_rate,
        "saved_examples": saved_examples,
    }

In [10]:
# Same normalization as your CIFAR test loader
CIFAR_MEAN = torch.tensor([0.485, 0.456, 0.406])
CIFAR_STD = torch.tensor([0.229, 0.224, 0.225])

In [11]:
num_classes = 100
dataset = '100'

In [ ]:
load_path = f'/content/drive/MyDrive/DLBA_Project/Models/vgg16_cifar{dataset}_two_stage_finetune.pth' # for fine tuned model only
dense_model = get_model(num_classes=num_classes, path=load_path)
trainset, trainloader, testset, testloader, valset, valloader, classes = load_dataset(dataset = dataset)
pruned_load_path_string = f'/content/drive/MyDrive/DLBA_Project/Models/vgg16_cifar{dataset}_prune_finetune_'
model_sparsities = [20,40,60,80]
# epsilons = [2/255, 4/255, 8/255, 16/255]
epsilons = [2/255]
for sparsity in model_sparsities:
  pruned_load_path = pruned_load_path_string+str(sparsity)+".pth"
  pruned_model = get_model(num_classes=num_classes, path=pruned_load_path)
  for epsilon in epsilons:
    print(f"Sparsity: {sparsity}, Epsilon: {epsilon}")
    dense_to_pruned = evaluate_transfer_attack(
      source_model=dense_model,
      target_model=pruned_model,
      testloader=testloader,
      epsilon=epsilon,
      device=device,
      save_per_batch=0,
      mean=CIFAR_MEAN,
      std=CIFAR_STD
    )
    print("Dense -> Pruned transfer")
    print(f"Source clean acc: {dense_to_pruned['source_clean_acc']:.4f}")
    print(f"Target clean acc: {dense_to_pruned['target_clean_acc']:.4f}")
    print(f"Source adv acc:   {dense_to_pruned['source_adv_acc']:.4f}")
    print(f"Target adv acc:   {dense_to_pruned['target_adv_acc']:.4f}")
    print(f"Transfer fooling rate: {dense_to_pruned['target_transfer_fooling_rate']:.4f}")

    # Pruned -> Dense
    pruned_to_dense = evaluate_transfer_attack(
        source_model=pruned_model,
        target_model=dense_model,
        testloader=testloader,
        epsilon=epsilon,
        device=device,
        save_per_batch=0,
        mean=CIFAR_MEAN,
        std=CIFAR_STD
    )

    print("\nPruned -> Dense transfer")
    print(f"Source clean acc: {pruned_to_dense['source_clean_acc']:.4f}")
    print(f"Target clean acc: {pruned_to_dense['target_clean_acc']:.4f}")
    print(f"Source adv acc:   {pruned_to_dense['source_adv_acc']:.4f}")
    print(f"Target adv acc:   {pruned_to_dense['target_adv_acc']:.4f}")
    print(f"Transfer fooling rate: {pruned_to_dense['target_transfer_fooling_rate']:.4f}")




Loading model from path /content/drive/MyDrive/DLBA_Project/Models/vgg16_cifar100_two_stage_finetune.pth
CIFAR-100 dataset downloaded and loaded successfully.
Loading model from path /content/drive/MyDrive/DLBA_Project/Models/vgg16_cifar100_prune_finetune_20.pth
Sparsity: 20, Epsilon: 0.00784313725490196


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
# Dense -> Pruned
dense_to_pruned = evaluate_transfer_attack(
    source_model=dense_model,
    target_model=pruned_model,
    testloader=testloader,
    epsilon=epsilon,
    device=device,
    save_per_batch=3
)

print("Dense -> Pruned transfer")
print(f"Source clean acc: {dense_to_pruned['source_clean_acc']:.4f}")
print(f"Target clean acc: {dense_to_pruned['target_clean_acc']:.4f}")
print(f"Source adv acc:   {dense_to_pruned['source_adv_acc']:.4f}")
print(f"Target adv acc:   {dense_to_pruned['target_adv_acc']:.4f}")
print(f"Transfer fooling rate: {dense_to_pruned['target_transfer_fooling_rate']:.4f}")


# Pruned -> Dense
pruned_to_dense = evaluate_transfer_attack(
    source_model=pruned_model,
    target_model=dense_model,
    testloader=testloader,
    epsilon=epsilon,
    device=device,
    save_per_batch=3
)

print("\nPruned -> Dense transfer")
print(f"Source clean acc: {pruned_to_dense['source_clean_acc']:.4f}")
print(f"Target clean acc: {pruned_to_dense['target_clean_acc']:.4f}")
print(f"Source adv acc:   {pruned_to_dense['source_adv_acc']:.4f}")
print(f"Target adv acc:   {pruned_to_dense['target_adv_acc']:.4f}")
print(f"Transfer fooling rate: {pruned_to_dense['target_transfer_fooling_rate']:.4f}")